# Prediction of Students at Risk of Academic Failure
## Early Warning System

This notebook implements the workflow defined in the specifications. It compares traditional Machine Learning approaches (Logistic Regression, Random Forest, SVM) with Deep Learning architectures (Artificial Neural Networks using PyTorch) to predict academic failure early.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix, 
                             recall_score, precision_score, f1_score, 
                             accuracy_score, roc_auc_score, roc_curve, ConfusionMatrixDisplay)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from imblearn.over_sampling import SMOTE

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import warnings
warnings.filterwarnings('ignore')

## Phase 1: Exploration and Preparation
Loading the dataset, verifying structure, categorical encoding, and train-test splits.

In [2]:
# 1. Load Data
df = pd.read_csv('risk_students.csv')
print("Dataset Shape:", df.shape)
print("\nMissing Values:\n", df.isnull().sum())

# Explore class distribution
print("\nClass Distribution:")
print(df['At_Risk'].value_counts(normalize=True) * 100)

display(df.head())

Dataset Shape: (2000, 11)

Missing Values:
 Age                              0
Gender                           0
Socio_Economic_Status            0
GPA                              0
Attendance_Rate                  0
Assignments_Submitted            0
Online_Learning_Hours            0
Forum_Participation              0
Lab_Performance                  0
Extracurricular_Participation    0
At_Risk                          0
dtype: int64

Class Distribution:
At_Risk
0    71.15
1    28.85
Name: proportion, dtype: float64


,Age,Gender,Socio_Economic_Status,GPA,Attendance_Rate,Assignments_Submitted,Online_Learning_Hours,Forum_Participation,Lab_Performance,Extracurricular_Participation,At_Risk
0,24,Female,Low,3.23,98.1,8,8.1,2,59,0,0
1,21,Female,High,3.82,81.4,6,0.4,6,73,0,0
2,28,Female,High,3.02,59.2,4,1.2,1,48,1,1
3,25,Male,Low,3.00,55.3,4,6.3,1,14,1,1
4,22,Female,High,2.10,90.6,16,14.1,2,82,0,0


In [3]:
# 2. Categorical Encoding (Binary and Ordinal)
df['Gender'] = df['Gender'].map({'Female': 0, 'Male': 1})
df['Socio_Economic_Status'] = df['Socio_Economic_Status'].map({'Low': 0, 'Medium': 1, 'High': 2})

# Separate features and target
X = df.drop('At_Risk', axis=1)
y = df['At_Risk']

# 3. Train-Test Split (80/20 with Stratification)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)

# Feature Scaling (StandardScaler)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training set shape:", X_train_scaled.shape)
print("Testing set shape:", X_test_scaled.shape)

Training set shape: (1600, 10)
Testing set shape: (400, 10)


## Phase 2: Machine Learning Modeling
Managing class imbalance via SMOTE and evaluating standard classifiers.

In [4]:
# Apply SMOTE to training data only to prevent data leakage
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_scaled, y_train)

print("Before SMOTE class distribution:\n", y_train.value_counts())
print("\nAfter SMOTE class distribution:\n", y_train_sm.value_counts())

Before SMOTE class distribution:
 At_Risk
0    1138
1     462
Name: count, dtype: int64

After SMOTE class distribution:
 At_Risk
0    1138
1    1138
Name: count, dtype: int64


In [5]:
# Initialize Models
ml_models = {
    'Logistic Regression': LogisticRegression(C=1.0, penalty='l2', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
    'SVM': SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
}

results = {}
predictions = {} # Store true vs pred for advanced plotting
pred_probs = {}  # Store probabilities for ROC curves

# Train and Evaluate
for name, model in ml_models.items():
    model.fit(X_train_sm, y_train_sm)
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    
    # Target metrics
    acc = accuracy_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)
    
    results[name] = {'Accuracy': acc, 'Recall': rec, 'Precision': prec, 'F1-Score': f1, 'ROC-AUC': roc_auc}
    predictions[name] = y_pred
    pred_probs[name] = y_prob
    
    print(f"--- {name} ---")
    print(classification_report(y_test, y_pred))
    print("\n")

--- Logistic Regression ---
              precision    recall  f1-score   support

           0       0.87      0.68      0.76       285
           1       0.48      0.74      0.58       115

    accuracy                           0.70       400
   macro avg       0.67      0.71      0.67       400
weighted avg       0.76      0.70      0.71       400



--- Random Forest ---
              precision    recall  f1-score   support

           0       0.87      0.73      0.79       285
           1       0.52      0.73      0.61       115

    accuracy                           0.73       400
   macro avg       0.70      0.73      0.70       400
weighted avg       0.77      0.73      0.74       400



--- SVM ---
              precision    recall  f1-score   support

           0       0.85      0.69      0.76       285
           1       0.47      0.69      0.56       115

    accuracy                           0.69       400
   macro avg       0.66      0.69      0.66       400
weighted

## Phase 3: Deep Learning Modeling (PyTorch)
Evaluating Artificial Neural Networks with class weighting on the BCE loss to manage imbalance.

In [6]:
# Setup Device and PyTorch Datasets
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

# Convert scaled numpy arrays to Tensors
X_train_t = torch.FloatTensor(X_train_scaled).to(device)
y_train_t = torch.FloatTensor(y_train.values).unsqueeze(1).to(device)
X_test_t = torch.FloatTensor(X_test_scaled).to(device)
y_test_t = torch.FloatTensor(y_test.values).unsqueeze(1).to(device)

# Create DataLoaders
train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Define Loss function with explicit Class Weights (pos_weight) for Imbalance
num_pos = y_train.sum()
num_neg = len(y_train) - num_pos
pos_weight = torch.tensor([num_neg / num_pos], dtype=torch.float32).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

input_dim = X_train_t.shape[1]

Using device: cpu


In [7]:
# Define Architectures

# Model 1: Baseline
class ANN_Baseline(nn.Module):
    def __init__(self, input_dim):
        super(ANN_Baseline, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )
    def forward(self, x):
        return self.net(x)

# Model 2: Deep 
class ANN_Deep(nn.Module):
    def __init__(self, input_dim):
        super(ANN_Deep, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1)
        )
    def forward(self, x):
        return self.net(x)

# Model 3: Regularized (Dropout)
class ANN_Regularized(nn.Module):
    def __init__(self, input_dim):
        super(ANN_Regularized, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.Dropout(0.3),
            nn.Tanh(),          # Using Tanh as specified in the report
            nn.Linear(32, 16),
            nn.Dropout(0.3),
            nn.Tanh(),
            nn.Linear(16, 8),
            nn.Dropout(0.3),
            nn.Tanh(),
            nn.Linear(8, 1)
        )
    def forward(self, x):
        return self.net(x)

In [8]:
# Reusable Training & Evaluation Function
def train_and_eval_model(model, optimizer, name, epochs=100):
    # Training loop
    model.train()
    for epoch in range(epochs):
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
    # Evaluation
    model.eval()
    with torch.no_grad():
        preds_logits = model(X_test_t)
        # BCEWithLogitsLoss utilizes unactivated outputs, apply sigmoid for probability
        preds_prob = torch.sigmoid(preds_logits).squeeze().cpu().numpy()
        preds = (preds_prob >= 0.5).astype(int)
        
    print(f"--- {name} ---")
    print(classification_report(y_test, preds))
    print("\n")
    
    # Save metrics globally so we can plot them easily
    results[name] = {
        'Accuracy': accuracy_score(y_test, preds),
        'Recall': recall_score(y_test, preds),
        'Precision': precision_score(y_test, preds),
        'F1-Score': f1_score(y_test, preds),
        'ROC-AUC': roc_auc_score(y_test, preds_prob)
    }
    
    predictions[name] = preds
    pred_probs[name] = preds_prob

# Train Model 1
m1 = ANN_Baseline(input_dim).to(device)
opt1 = optim.SGD(m1.parameters(), lr=0.01)
train_and_eval_model(m1, opt1, 'ANN 1 (Baseline)', epochs=100)

# Train Model 2
m2 = ANN_Deep(input_dim).to(device)
opt2 = optim.Adam(m2.parameters(), lr=0.001)
train_and_eval_model(m2, opt2, 'ANN 2 (Deep)', epochs=100)

# Train Model 3
m3 = ANN_Regularized(input_dim).to(device)
opt3 = optim.Adam(m3.parameters(), lr=0.001)
train_and_eval_model(m3, opt3, 'ANN 3 (Regularized)', epochs=150)

--- ANN 1 (Baseline) ---
              precision    recall  f1-score   support

           0       0.87      0.70      0.77       285
           1       0.49      0.73      0.59       115

    accuracy                           0.71       400
   macro avg       0.68      0.71      0.68       400
weighted avg       0.76      0.71      0.72       400



--- ANN 2 (Deep) ---
              precision    recall  f1-score   support

           0       0.85      0.71      0.77       285
           1       0.49      0.68      0.57       115

    accuracy                           0.70       400
   macro avg       0.67      0.70      0.67       400
weighted avg       0.74      0.70      0.71       400



--- ANN 3 (Regularized) ---
              precision    recall  f1-score   support

           0       0.88      0.72      0.79       285
           1       0.52      0.76      0.62       115

    accuracy                           0.73       400
   macro avg       0.70      0.74      0.71       

## Phase 4: Final Evaluation
Reviewing metrics prioritizing **Recall** to minimize false negatives (failing to identify at-risk students).

In [9]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.cm as cm

# 1. Compile Results into a DataFrame
final_results_df = pd.DataFrame(results).T
print("Model Evaluation Rankings (Sorted by Recall):")
display(final_results_df.sort_values(by='Recall', ascending=False).style.background_gradient(cmap='Greens'))

def plot_bar_chart():
    final_results_df.plot(kind='bar', figsize=(14, 6), colormap='viridis', width=0.8)
    plt.title('Performance Metrics Comparison Across All Models', fontsize=15, fontweight='bold')
    plt.ylabel('Score (0.0 to 1.0)', fontsize=12)
    plt.xticks(rotation=45, ha='right', fontsize=11)
    plt.legend(loc='lower center', bbox_to_anchor=(0.5, -0.3), ncol=5)
    plt.ylim(0, 1.1)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

def plot_roc_curves():
    colors = cm.tab10(np.linspace(0, 1, len(pred_probs)))
    plt.figure(figsize=(10, 8))
    for (name, y_prob), color in zip(pred_probs.items(), colors):
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        auc_score = results[name]['ROC-AUC']
        plt.plot(fpr, tpr, label=f'{name} (AUC = {auc_score:.3f})', linewidth=2.5, color=color)

    plt.plot([0, 1], [0, 1], 'k--', label='Random Chance', linewidth=2)
    plt.title('Receiver Operating Characteristic (ROC) Curves', fontsize=15, fontweight='bold')
    plt.xlabel('False Positive Rate (Fall-out)', fontsize=12)
    plt.ylabel('True Positive Rate (Recall)', fontsize=12)
    plt.legend(loc='lower right', fontsize=10)
    plt.grid(alpha=0.5)
    plt.tight_layout()
    plt.show()

def plot_confusion_matrices():
    fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(16, 10))
    axes = axes.flatten()

    for idx, (name, y_pred) in enumerate(predictions.items()):
        ax = axes[idx]
        cm_matrix = confusion_matrix(y_test, y_pred)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm_matrix, display_labels=['Not At Risk', 'At Risk'])
        disp.plot(ax=ax, cmap='Blues', colorbar=False, values_format='d')
        
        recall_val = results[name]["Recall"]
        prec_val = results[name]["Precision"]
        ax.set_title(f'{name}\nRecall: {recall_val:.2f} | Precision: {prec_val:.2f}', fontsize=12, fontweight='bold')
        ax.grid(False)

    plt.suptitle('Confusion Matrices: False Negatives vs. False Positives', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Create interactive selection
def interactive_plot(view_type):
    if view_type == 'Bar Chart':
        plot_bar_chart()
    elif view_type == 'ROC Curves':
        plot_roc_curves()
    elif view_type == 'Confusion Matrices':
        plot_confusion_matrices()

# Display dropdown menu
dropdown = widgets.Dropdown(
    options=['Bar Chart', 'ROC Curves', 'Confusion Matrices'],
    value='Bar Chart',
    description='Select View:',
    style={'description_width': 'initial'}
)

widgets.interact(interactive_plot, view_type=dropdown);

Model Evaluation Rankings (Sorted by Recall):


,Accuracy,Recall,Precision,F1-Score,ROC-AUC
ANN 3 (Regularized),0.732500,0.756522,0.524096,0.619217,0.811655
Logistic Regression,0.697500,0.739130,0.482955,0.584192,0.771381
Random Forest,0.730000,0.730435,0.521739,0.608696,0.824439
ANN 1 (Baseline),0.707500,0.730435,0.494118,0.589474,0.797925
SVM,0.690000,0.686957,0.473054,0.560284,0.770679
ANN 2 (Deep),0.702500,0.678261,0.487500,0.567273,0.744134


interactive(children=(Dropdown(description='Select View:', options=('Bar Chart', 'ROC Curves', 'Confusion Matr…